In [3]:
import pandas as pd
import numpy as np

df = pd.read_excel("public_procedural_corpus_500_mixed.xlsx")

In [4]:
df.columns
df.head()

,sentence,obligation,conditional,cross_reference,exception,long_sentence,source_tag,discretion,friction_score_v2,feature_combo_v2
0,of The laboratory should establish rejection c...,1,0,0,0,0,WHO,0,0,0-0-0-0
1,laboratory should periodically provide trainin...,1,0,0,0,1,WHO,0,0,0-0-0-0
2,The laboratory must make available a test requ...,1,0,0,0,1,WHO,0,0,0-0-0-0
3,/departments should conduct a risk assessment ...,1,0,0,0,0,WHO,0,0,0-0-0-0
4,must be kept up-to-date and be referenced in t...,1,0,1,0,0,WHO,0,1,0-1-0-0


In [5]:
features = ["conditional", "cross_reference", "exception", "discretion"]

df[features].mean().round(3)

conditional        0.338
cross_reference    0.118
exception          0.036
discretion         0.120
dtype: float64

In [6]:
(df[features].mean()*100).round(1)

conditional        33.8
cross_reference    11.8
exception           3.6
discretion         12.0
dtype: float64

In [7]:
df["friction_score_v2"].value_counts().sort_index()

friction_score_v2
0    251
1    194
2     53
3      2
Name: count, dtype: int64

In [8]:
(df["friction_score_v2"].value_counts().sort_index()/len(df)*100).round(1)

friction_score_v2
0    50.2
1    38.8
2    10.6
3     0.4
Name: count, dtype: float64

In [9]:
df.groupby("source_tag")["friction_score_v2"].mean().round(2)

source_tag
BSL3         0.47
WHO          0.50
chemical     0.60
synthetic    0.95
Name: friction_score_v2, dtype: float64

In [10]:
df.sort_values("friction_score_v2", ascending=False)[
    ["sentence","conditional","cross_reference","exception","discretion","friction_score_v2"]
].head(10)

,sentence,conditional,cross_reference,exception,discretion,friction_score_v2
498,"If reporting is delayed, the analyst shall not...",1,1,0,1,3
491,"When contamination is confirmed, the operator ...",1,1,1,0,3
499,"When retesting is required, the technician mus...",1,0,0,1,2
495,"When drift exceeds tolerance, the technician m...",1,0,1,0,2
496,"If precipitation persists, the analyst shall f...",1,1,0,0,2
494,"If instability occurs, the operator shall repl...",1,0,0,1,2
451,The operator must continue except when alarms ...,1,0,1,0,2
456,Analysis shall continue except when interferen...,1,0,1,0,2
493,"When acceptance criteria are unmet, the analys...",1,0,0,1,2
450,The analyst shall proceed unless otherwise ins...,1,0,1,0,2


In [11]:
import pandas as pd

# ===== 1) Friction score distribution =====
dist = (
    df["friction_score_v2"]
    .value_counts()
    .sort_index()
    .reset_index()
)
dist.columns = ["friction_score", "count"]
dist.to_csv("cf_friction_distribution.csv", index=False)

# ===== 2) Feature counts =====
features = ["conditional", "cross_reference", "exception", "discretion"]

feat_counts = df[features].sum().reset_index()
feat_counts.columns = ["feature", "count"]
feat_counts.to_csv("cf_feature_counts.csv", index=False)

# ===== 3) Co-occurrence matrix =====
co = pd.DataFrame(index=features, columns=features)

for a in features:
    for b in features:
        co.loc[a, b] = int(((df[a] == 1) & (df[b] == 1)).sum())

co.to_csv("cf_feature_cooccurrence.csv")

print("Exported: cf_friction_distribution.csv, cf_feature_counts.csv, cf_feature_cooccurrence.csv")

Exported: cf_friction_distribution.csv, cf_feature_counts.csv, cf_feature_cooccurrence.csv


In [12]:
df.columns

Index(['sentence', 'obligation', 'conditional', 'cross_reference', 'exception',
       'long_sentence', 'source_tag', 'discretion', 'friction_score_v2',
       'feature_combo_v2'],
      dtype='object')

In [13]:
import os
os.getcwd()

'C:\\Users\\user'

In [14]:
import pandas as pd
import numpy as np

df = pd.read_excel("public_procedural_corpus_500_mixed.xlsx")

features = ["conditional", "cross_reference", "exception", "discretion"]

# ── 1. Natural sentences only ──────────────────────────────────────────────
df_nat = df[df["source_tag"] != "synthetic"]
print(f"N (natural only): {len(df_nat)}")  # 확인용, 400이어야 함

# ── 2. Feature frequency ───────────────────────────────────────────────────
print("\n[Feature frequency — natural only]")
print((df_nat[features].mean() * 100).round(1))

print("\n[Feature frequency — full corpus]")
print((df[features].mean() * 100).round(1))

# ── 3. Friction score distribution ────────────────────────────────────────
print("\n[Friction score distribution — natural only (%)]")
print((df_nat["friction_score_v2"].value_counts().sort_index() / len(df_nat) * 100).round(1))

print("\n[Friction score distribution — full corpus (%)]")
print((df["friction_score_v2"].value_counts().sort_index() / len(df) * 100).round(1))

# ── 4. Mean friction by source (natural only) ─────────────────────────────
print("\n[Mean friction by source — natural only]")
print(df_nat.groupby("source_tag")["friction_score_v2"].mean().round(2))

# ── 5. Co-occurrence matrix — natural only ────────────────────────────────
co_nat = pd.DataFrame(index=features, columns=features)
for a in features:
    for b in features:
        co_nat.loc[a, b] = int(((df_nat[a] == 1) & (df_nat[b] == 1)).sum())

print("\n[Co-occurrence matrix — natural only]")
print(co_nat)

# ── 6. Export ──────────────────────────────────────────────────────────────
dist_nat = (
    df_nat["friction_score_v2"]
    .value_counts().sort_index()
    .reset_index()
)
dist_nat.columns = ["friction_score", "count"]
dist_nat.to_csv("cf_friction_distribution_natural.csv", index=False)

feat_nat = df_nat[features].sum().reset_index()
feat_nat.columns = ["feature", "count"]
feat_nat.to_csv("cf_feature_counts_natural.csv", index=False)

co_nat.to_csv("cf_feature_cooccurrence_natural.csv")

print("\nExported: natural-only CSVs")

N (natural only): 400

[Feature frequency — natural only]
conditional        26.2
cross_reference    11.8
exception           1.5
discretion         13.2
dtype: float64

[Feature frequency — full corpus]
conditional        33.8
cross_reference    11.8
exception           3.6
discretion         12.0
dtype: float64

[Friction score distribution — natural only (%)]
friction_score_v2
0    54.2
1    38.8
2     7.0
Name: count, dtype: float64

[Friction score distribution — full corpus (%)]
friction_score_v2
0    50.2
1    38.8
2    10.6
3     0.4
Name: count, dtype: float64

[Mean friction by source — natural only]
source_tag
BSL3        0.47
WHO         0.50
chemical    0.60
Name: friction_score_v2, dtype: float64

[Co-occurrence matrix — natural only]
                conditional cross_reference exception discretion
conditional             105              13         2          9
cross_reference          13              47         0          4
exception                 2               0   